# Rule Extraction Engine — prototype

**Question:** given the IU matriculation handbook and a free-text program description, can an LLM pipeline author admission rules that 
- compile through the real `app/rules_engine` compiler 
- reproduce the hand-authored policy's decisions on the 16 saved gold applicants?

**Architecture** (each stage is a cell; every intermediate is printed and saved to `artifacts/`):

```
handbook.md ──> [0 section index] ──> [1 chunkless retrieval]    (agentic navigation, no chunks/embeddings)
                                        │ opened sections
program description ──────────────────> [2 requirements extraction]   (structured, with citations)
                                        │ requirements.json
engine vocabulary (introspected) ─────> [3 vocabulary mapping]        (supported vs UNSUPPORTED report)
                                        │ mapping.json
  rules/README.md (DSL spec) ───┐
                                ├─────> [4 YAML generation]           (structure-only skeleton, no policy content)
  structure-only skeleton ──────┘         │ generated-rules-c/
                                        [5 compile + bounded repair]  (RulesEngine.activate is the guardrail)
                                        │ activated engine
                                        [6 gold evaluation]           (16 saved applicants vs hand-authored baseline)
```

**Honesty note:** `rules/README.md` (the DSL spec, which the generator must see) embeds the hand-authored `GERMAN_ABITUR` rule as its worked example. The other four rules, the shared requirements, and the policy resolution are still derived from the handbook.

In [23]:
# PROTOTYPE — throwaway demo, not production code.
import json
import os
import re
import shutil
import sys
from datetime import datetime
from pathlib import Path

from pydantic import BaseModel

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "rules").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "rules").is_dir(), "run this notebook from the project root or any folder inside it"
sys.path.insert(0, str(PROJECT_ROOT))

RULES_DIR = PROJECT_ROOT / "rules"
HANDBOOK = PROJECT_ROOT / "case-study" / "IU-FS-LF-Leitfaden-Hochschulzugangsberechtigung-Stand-Januar2025.md"
RUNS_DIR = PROJECT_ROOT / "runs"          # gold applicant bundles, read by stage 6
OUT_DIR = PROJECT_ROOT / "tools" / "rule-extractor"

# Everything this run produces goes in one timestamped folder, so a run never overwrites
# an earlier one and a trace can always be matched to the package it produced.
RUN_DIR = RUNS_DIR / f"generate-rules-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
ARTIFACTS_DIR = RUN_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Pick up OPENAI_API_KEY from the project .env if the shell did not export it.
env_file = PROJECT_ROOT / ".env"
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip().strip('"'))

from openai import OpenAI

MODEL = os.environ.get("RULE_EXTRACTOR_MODEL", "gpt-5.4-mini")
client = OpenAI()


def ask(instructions: str, user_input: str, schema: type[BaseModel]) -> BaseModel:
    """One structured-output call; the schema is enforced by the API."""
    response = client.responses.parse(
        model=MODEL,
        instructions=instructions,
        input=user_input,
        text_format=schema,
        store=False,
    )
    if response.status != "completed":
        raise RuntimeError(f"model response status: {response.status}")
    return response.output_parsed


def save_artifact(name: str, payload) -> None:
    (ARTIFACTS_DIR / name).write_text(json.dumps(payload, indent=2, ensure_ascii=False))


print(f"project: {PROJECT_ROOT}")
print(f"model:   {MODEL}")
print(f"run dir: {RUN_DIR.relative_to(PROJECT_ROOT)}/")

project: /Users/pratikpatil/Code/auto admissions codex
model:   gpt-5.4-mini
run dir: runs/generate-rules-20260827-142343/


## Stage 0 — Ingest the handbook and build the section index

The handbook markdown stands in for the uploaded PDF (the English conversion already exists; PDF→text is out of demo scope). We parse the `##`/`###` heading tree into a numbered index — **this index is the only thing the retrieval stage is allowed to see up front**. The document's own structure is the retrieval index: no chunking, no embeddings.

The inventory is ported from `alternate-options/option-cd-agentic-rag/src/policy_index.py`, so this experiment and options C/D retrieve over **provably the same substrate** (137 openable sections of 142 headings; the notebook asserts nothing, but the two implementations were diffed and agree exactly). Three properties matter:

- **Non-overlapping spans.** A heading runs to the *next* heading of any level. Previously a `##` chapter's span swallowed its `###` children, so opening a chapter and one of its subsections sent the same text twice.
- **Breadcrumbs.** A `###` carries its parent `##` chapter, so `Anabin` reads as `GENERAL SPECIAL FEATURES OF FOREIGN QUALIFICATIONS › Anabin`.
- **Heading-only shells dropped.** A `##` that is nothing but its own heading line has nothing to retrieve; it survives only as its children's breadcrumb.

Line counts on each entry are this notebook's one addition to the option C/D rendering — the navigator has a 3-turn budget and needs to see which chapters are bulk. Without them it spent a whole turn opening all 24 per-programme subsections of `SPECIAL REQUIREMENTS IN BA DEGREE PROGRAMS`.

In [24]:
assert HANDBOOK.exists(), f"{HANDBOOK}\nPut the English markdown handbook at this path. It is IU material and is not in the repository: see case-study/README.md."
handbook_text = HANDBOOK.read_text()

# Section inventory, ported from alternate-options/option-cd-agentic-rag/src/policy_index.py
# so this experiment and options C/D retrieve over the identical substrate:
#   - spans are non-overlapping (a heading runs to the NEXT heading of any level), so opening
#     a chapter and one of its subsections no longer sends the same text twice;
#   - a ### section carries its parent ## chapter as a breadcrumb, so "Anabin" is not ambiguous;
#   - a ## that is nothing but its own heading line has nothing to retrieve and is dropped from
#     the openable inventory — it survives only as its children's breadcrumb;
#   - ids number ALL headings in document order, so an id keeps meaning the same section.
HEADING_RE = re.compile(r"^(#{2,3}) (.+?)\s*$", re.MULTILINE)

matches = list(HEADING_RE.finditer(handbook_text))
all_sections = []
chapter_title = None
for index, match in enumerate(matches):
    level = len(match.group(1))
    if level == 2:
        chapter_title = match.group(2)
    all_sections.append({
        "id": index,
        "level": level,
        "title": match.group(2),
        "breadcrumb": chapter_title if level == 3 else None,
        "start": match.start(),
        "end": matches[index + 1].start() if index + 1 < len(matches) else len(handbook_text),
    })
by_id = {s["id"]: s for s in all_sections}


def section_text(section_id: int) -> str:
    section = by_id[section_id]
    return handbook_text[section["start"]:section["end"]].rstrip()


def full_title(section_id: int) -> str:
    section = by_id[section_id]
    return f"{section['breadcrumb']} \u203a {section['title']}" if section["breadcrumb"] else section["title"]


sections = [s for s in all_sections if section_text(s["id"]).strip() != f"## {s['title']}"]

# Line counts are this notebook's one addition to the option C/D rendering. The navigator has
# a 3-turn budget, and without a size signal it spent turn 3 opening all 24 per-programme
# subsections of SPECIAL REQUIREMENTS IN BA DEGREE PROGRAMS. The section SET stays identical
# to option C/D's; only the label carries more.
toc_listing = "\n".join(
    f"[{s['id']:>3}] {full_title(s['id'])}  ({len(section_text(s['id']).splitlines())} lines)"
    for s in sections
)
print(f"{len(sections)} openable sections of {len(all_sections)} headings, "
      f"{len(handbook_text.splitlines())} handbook lines\n")
print(toc_listing)


137 openable sections of 142 headings, 6313 handbook lines

[  0] CONTENTS  (245 lines)
[  1] CHANGE TRACKING  (169 lines)
[  3] ACCESS TO THE BACHELOR STUDY PROGRAM › Studying with the (Fach-) Abitur (general or subject-restricted school-leaving qualification)  (150 lines)
[  4] ACCESS TO THE BACHELOR STUDY PROGRAM › Studying without the (Fach-)Abitur (general or subject-restricted school-leaving qualification)  (263 lines)
[  6] GENERAL SPECIAL FEATURES OF FOREIGN QUALIFICATIONS › Anabin  (5 lines)
[  7] GENERAL SPECIAL FEATURES OF FOREIGN QUALIFICATIONS › Certificate of equivalence (Äquivalenzbescheinigung)  (10 lines)
[  8] PROOF OF LANGUAGE PROFICIENCY - GERMAN LANGUAGE SKILLS  (32 lines)
[  9] PROOF OF LANGUAGE PROFICIENCY - GERMAN LANGUAGE SKILLS › Luxembourg  (10 lines)
[ 11] ACCESS TO THE BACHELOR'S DEGREE PROGRAM FOR APPLICANTS FROM AUSTRIA › General higher education entrance qualification / Matura  (4 lines)
[ 12] ACCESS TO THE BACHELOR'S DEGREE PROGRAM FOR APPLICANTS FROM A

In [25]:
# The user's free-text input: which program to extract rules for.
PROGRAM_DESCRIPTION = (
    "Bachelor's degree program at IU International University of Applied Sciences "
    "Extract the admission eligibility rules that determine whether an applicant "
    "may access a Bachelor's study program."
)
COMPILE_AND_EVALUATE = True  # False: author + save the rule files only (no compile guardrail, no gold eval)
POLICY_ID = "IU_BACHELOR_ACCESS"
POLICY_VERSION = "0.0.22"
STUDY_LEVEL = "BACHELOR"
POLICY_FILE = "bachelors-access.yaml"
EXPECTED_FILES = {POLICY_FILE, "school-access-rules.yaml", "professional-access-rules.yaml",
                  "common/requirements.yaml", "common/conditions.yaml"}

print(PROGRAM_DESCRIPTION)
print(f"mode: {'compile + gold eval' if COMPILE_AND_EVALUATE else 'save files for human review only'}")
print(f"output: {RUN_DIR.relative_to(PROJECT_ROOT)}/")


Bachelor's degree program at IU International University of Applied Sciences Extract the admission eligibility rules that determine whether an applicant may access a Bachelor's study program.
mode: compile + gold eval
output: runs/generate-rules-20260827-142343/


## Stage 1 — Chunkless retrieval: agentic navigation

The LLM sees only the ToC and the program description. Each turn it requests sections to open (by id), reads them, and may request more — until it declares coverage complete or the turn budget runs out. The navigation trace below is the retrieval rationale, fully inspectable.

In [26]:
MAX_NAV_TURNS = 3


class NavigationTurn(BaseModel):
    rationale: str
    open_section_ids: list[int]
    coverage_complete: bool


NAV_INSTRUCTIONS = (
    "You are the retrieval stage of an admissions rule extraction engine. "
    "You navigate a policy handbook using only its table of contents — no chunking, no embeddings. "
    "Goal: open exactly the sections needed to author machine-readable admission eligibility "
    "rules for the described study program. Each turn, request section ids to open; you will see "
    "the full text of every opened section on the next turn. Set coverage_complete=true only when "
    "the opened sections fully cover admission eligibility for the described program. "
    "Do not open sections irrelevant to eligibility (change logs, other study levels, formalities)."
)

opened: dict[int, str] = {}
navigation_trace = []
for turn in range(1, MAX_NAV_TURNS + 1):
    opened_blob = "\n\n".join(
        f"=== [{sid}] {full_title(sid)} ===\n{text}" for sid, text in opened.items()
    ) or "(none yet)"
    nav = ask(
        NAV_INSTRUCTIONS,
        f"PROGRAM DESCRIPTION:\n{PROGRAM_DESCRIPTION}\n\n"
        f"TABLE OF CONTENTS:\n{toc_listing}\n\n"
        f"OPENED SECTIONS SO FAR:\n{opened_blob}",
        NavigationTurn,
    )
    openable_ids = {s["id"] for s in sections}
    new_ids = [i for i in nav.open_section_ids if i in openable_ids and i not in opened]
    for sid in new_ids:
        opened[sid] = section_text(sid)
    navigation_trace.append({
        "turn": turn,
        "rationale": nav.rationale,
        "opened": [by_id[i]["title"] for i in new_ids],
        "coverage_complete": nav.coverage_complete,
    })
    print(f"— turn {turn} —")
    print(f"  rationale: {nav.rationale}")
    for sid in new_ids:
        print(f"  opened [{sid}] {full_title(sid)}")
    if nav.coverage_complete and opened:
        print("  coverage declared complete")
        break

retrieved_blob = "\n\n".join(
    f"=== [{sid}] {full_title(sid)} ===\n{text}" for sid, text in opened.items()
)
save_artifact("navigation-trace.json", navigation_trace)
print(f"\nretrieved {len(opened)} sections, {len(retrieved_blob.splitlines())} lines total")

— turn 1 —
  rationale: To extract admission eligibility rules for a Bachelor's program, I need the core access rules for applicants with and without the Abitur, plus any foreign-qualification special cases and language/entrance-exam provisions that affect eligibility. I will avoid program-specific special requirements and non-eligibility sections unless needed.
  opened [3] ACCESS TO THE BACHELOR STUDY PROGRAM › Studying with the (Fach-) Abitur (general or subject-restricted school-leaving qualification)
  opened [4] ACCESS TO THE BACHELOR STUDY PROGRAM › Studying without the (Fach-)Abitur (general or subject-restricted school-leaving qualification)
  opened [6] GENERAL SPECIAL FEATURES OF FOREIGN QUALIFICATIONS › Anabin
  opened [7] GENERAL SPECIAL FEATURES OF FOREIGN QUALIFICATIONS › Certificate of equivalence (Äquivalenzbescheinigung)
  opened [8] PROOF OF LANGUAGE PROFICIENCY - GERMAN LANGUAGE SKILLS
  opened [17] BACHELOR ENTRANCE EXAMINATION (BACHELOR ZUGANGSPRÜFUNG)
  opened [4

## Stage 2 — Requirements extraction



In [27]:
class Citation(BaseModel):
    section_title: str
    quote: str


class Requirement(BaseModel):
    requirement_id: str
    summary: str
    conditions: str
    outcome: str
    blocking_information: str
    citations: list[Citation]


class RequirementSet(BaseModel):
    requirements: list[Requirement]


EXTRACT_INSTRUCTIONS = (
    "You extract admission eligibility requirements from policy text for later conversion into "
    "machine-readable rules. For every distinct admission path or requirement in the provided "
    "sections, record: an UPPER_SNAKE_CASE requirement_id; a one-sentence summary; the exact "
    "conditions, thresholds, and exceptions; the outcome the source prescribes when met "
    "(direct access, conditional access with e.g. a trial study or entrance exam, rejection); "
    "the information whose absence or uncertainty prevents a decision; and verbatim quotes with "
    "their section titles. Extract only what the source states — do not invent policy. "
    "Cover every admission path in the sections, including ones about vocational or "
    "professional qualifications, and note requirements that apply only in special cases."
)

requirement_set = ask(
    EXTRACT_INSTRUCTIONS,
    f"PROGRAM DESCRIPTION:\n{PROGRAM_DESCRIPTION}\n\nRETRIEVED SECTIONS:\n{retrieved_blob}",
    RequirementSet,
)

requirements_json = requirement_set.model_dump()
save_artifact("requirements.json", requirements_json)

print(f"{len(requirement_set.requirements)} requirements extracted\n")
for req in requirement_set.requirements:
    print(f"• {req.requirement_id}")
    print(f"    {req.summary}")
    print(f"    conditions: {req.conditions}")
    print(f"    outcome:    {req.outcome}")
    print(f"    blocked by: {req.blocking_information}")
    for cite in req.citations[:2]:
        print(f"    source:     [{cite.section_title}] \"{cite.quote[:110]}\"")
    print()

26 requirements extracted

• GENERAL_HZB_DIRECT_ACCESS
    Applicants with a recognized general higher education entrance qualification may be admitted directly to a Bachelor's program.
    conditions: Proof of the general higher education entrance qualification (Abitur / allgemeine Hochschulreife) is sufficient; the certificate must be in officially certified form in German or English. No special restriction applies.
    outcome:    Direct access.
    blocked by: Missing proof of the general higher education entrance qualification or missing officially certified German/English form prevents a decision.
    source:     [Studying with the (Fach-) Abitur (general or subject-restricted school-leaving qualification)] "Admission requirement: Proof of the general higher education entrance qualification, the subject-restricted hi"
    source:     [Abitur (German school-leaving qualification) › Allgemeine Hochschulreife (general higher education entrance qualification)] "The general higher edu

## Stage 3 — DSL Vocabulary mapping

The engine's authoring surface is **fixed in code**: five rule IDs, three candidate collections, a closed fact/enum vocabulary, four operators, and a reason-code catalog. We introspect it from the live code (no hardcoding), then ask the LLM to map each requirement onto it — or flag it as **unsupported**, stating the extension it would need. Silently dropping a policy requirement is the one unacceptable failure mode in admissions; the unsupported report is the honest output.

In [28]:
# Prototype: private compiler tables are the ground truth, so we import them directly.
from app.models.results import RULE_ORDER, ApplicationStatus, RuleStatus
from app.rules_engine.compiler import _APPLICATION_FACTS, _SOURCE_ALIAS, _SOURCE_FACTS
from app.rules_engine.reason_catalog import RULE_EXPLANATIONS


def fact_entry(spec) -> dict:
    return {
        "type": spec.kind.__name__,
        "allowed_values": sorted(spec.domain) if spec.domain else None,
    }


VOCAB = {
    "rule_ids": [rule.value for rule in RULE_ORDER],
    "rule_ids_note": "the policy must define exactly these five rule ids — no more, no fewer",
    "collections": {
        name: {"select_alias": _SOURCE_ALIAS[name], "facts": {k: fact_entry(v) for k, v in facts.items()}}
        for name, facts in _SOURCE_FACTS.items()
    },
    "application_facts": {k: fact_entry(v) for k, v in _APPLICATION_FACTS.items()},
    "operators": ["eq", "in", "gte", "lt", "all_of", "any_of", "ref"],
    "rule_statuses": [status.value for status in RuleStatus],
    "application_statuses": [status.value for status in ApplicationStatus],
    "reason_codes": {code: RULE_EXPLANATIONS[code] for code in sorted(RULE_EXPLANATIONS)},
    "reason_codes_note": "every reason_code in the YAML must be one of these exact keys",
}
VOCAB_TEXT = json.dumps(VOCAB, indent=1)
print(f"vocabulary introspected from the engine: {len(VOCAB['rule_ids'])} rule ids, "
      f"{sum(len(c['facts']) for c in VOCAB['collections'].values())} candidate facts, "
      f"{len(VOCAB['reason_codes'])} reason codes")

vocabulary introspected from the engine: 5 rule ids, 27 candidate facts, 29 reason codes


In [29]:
class RequirementMapping(BaseModel):
    requirement_id: str
    supported: bool
    target_rule_id: str | None
    facts_used: list[str]
    notes: str
    unsupported_reason: str | None


class MappingReport(BaseModel):
    mappings: list[RequirementMapping]


MAPPING_INSTRUCTIONS = (
    "You decide, for each extracted admission requirement, whether the deterministic rule engine "
    "can express it with its FIXED vocabulary (given below as JSON). A requirement is supported "
    "only if it can be evaluated using existing facts, operators, one of the five fixed rule ids, "
    "and existing reason codes. For supported requirements name the target rule id and the facts "
    "used. For unsupported requirements set supported=false and state precisely which engine "
    "extension would be needed (new fact, new collection, new rule id, new operator...). "
    "Never force a requirement onto facts that do not actually capture its meaning."
)

mapping_report = ask(
    MAPPING_INSTRUCTIONS,
    f"ENGINE VOCABULARY (JSON):\n{VOCAB_TEXT}\n\n"
    f"EXTRACTED REQUIREMENTS (JSON):\n{json.dumps(requirements_json, indent=1)}",
    MappingReport,
)

mapping_json = mapping_report.model_dump()
save_artifact("mapping.json", mapping_json)

supported = [m for m in mapping_report.mappings if m.supported]
unsupported = [m for m in mapping_report.mappings if not m.supported]

print(f"SUPPORTED ({len(supported)}):")
for m in supported:
    print(f"  • {m.requirement_id} -> {m.target_rule_id}  facts: {', '.join(m.facts_used)}")
print(f"\nUNSUPPORTED — engine extension needed ({len(unsupported)}):")
for m in unsupported:
    print(f"  • {m.requirement_id}: {m.unsupported_reason}")

SUPPORTED (11):
  • GENERAL_HZB_DIRECT_ACCESS -> GERMAN_ABITUR  facts: school_qualifications.qualification.type, school_qualifications.qualification.country, school_qualifications.qualification.completed, school_qualifications.qualification.validity_restriction_present, school_qualifications.qualification.validity_restriction_code
  • SUBJECT_RESTRICTED_HZB_FROM_GERMAN_SPEAKING_AREA_DIRECT_ACCESS -> FACHGEBUNDENE_HOCHSCHULREIFE  facts: school_qualifications.qualification.type, school_qualifications.qualification.country, school_qualifications.qualification.completed, school_qualifications.qualification.issuing_region
  • SUBJECT_RESTRICTED_HZB_FROM_ABROAD_TRIAL_STUDY -> FACHGEBUNDENE_HOCHSCHULREIFE  facts: school_qualifications.qualification.type, school_qualifications.qualification.country, school_qualifications.qualification.completed, school_qualifications.qualification.issuing_region
  • GENERAL_FHR_DIRECT_ACCESS_WITH_SCHOOL_AND_VOCATIONAL_PARTS -> GERMAN_GENERAL_FACHHOCHSCHULREIFE

## Stage 4 — YAML generation

The generator gets the DSL spec (`rules/README.md`), the introspected vocabulary, the two scaffold status files, the stage-2/3 artifacts, and a **structure-only skeleton**: every envelope field shown as a `<...>` placeholder, with zero policy content. It never sees the hand-authored rules.

Earlier runs measured two other arms and both were cut:

- **Arm A — spec only.** Never compiled, in any run. It guessed the file *envelope* wrong (sources field names, imports nesting, evaluation block) and `INVALID_POLICY_SCHEMA` is too opaque for the repair loop to steer by. The skeleton exists to target exactly this failure without handing over the answer key.
- **Arm B — spec + the hand-authored YAML.** Reproduced the reference near-verbatim every run. Its 16/16 measures copying, not derivation, so it was a copy-detection control rather than evidence.

What is left is the arm that can support a genuine *derived from the handbook* claim.

In [30]:
POLICY_SKELETON = r"""
# ============ SKELETON 1: policy entry file ============
dsl_version: "1.3"

policy:
  id: <POLICY_ID>
  version: "<version-string>"

  applies_when:
    fact: application.study_level
    eq: <STUDY_LEVEL_VALUE>

  sources:
    - file: <relative-path-to-source-document>
      section: <SECTION TITLE>
      subsections:
        - <Subsection title>

  imports:
    - namespace: rule_statuses
      file: rule-statuses.yaml
    - namespace: application_statuses
      file: application-statuses.yaml
    - namespace: requirements
      file: common/requirements.yaml
    - namespace: <module_namespace>
      file: <rule-module-file.yaml>

  evaluation:
    rule_groups:
      - include: <module_namespace>.<EXPORTED_RULE_GROUP_NAME>

  resolution:
    first_match:
      - when_any_rule:
          ref: rule_statuses.<RULE_STATUS>
        application_status:
          ref: application_statuses.<APPLICATION_STATUS>
      # ...one case per resolution priority, in order...
      - when_all_applicable_rules:
          ref: rule_statuses.<RULE_STATUS>
        application_status:
          ref: application_statuses.<APPLICATION_STATUS>
      - when_no_recognized_rule: true
        application_status:
          ref: application_statuses.<APPLICATION_STATUS>

# ============ SKELETON 2: rule module file (imported by the policy) ============
dsl_version: "1.3"

module:
  id: <MODULE_ID>
  version: "<version-string>"
  imports:
    - namespace: rule_statuses
      file: rule-statuses.yaml
    - namespace: conditions
      file: common/conditions.yaml
  requires_namespaces:
    - requirements

  exports:
    <RULE_GROUP_NAME>:
      id: <RULE_GROUP_NAME>
      rules:
        # body form 1: require + result
        - id: <RULE_ID>
          select:
            from: <collection_name>
            as: <collection_alias>
            where:
              fact: <alias>.<fact_name>
              eq: <VALUE>
          applicability:            # optional
            require:
              ref: requirements.<exported_requirement_name>
            result:
              not_applicable:
                status:
                  ref: rule_statuses.<RULE_STATUS>
                reason_code: <REASON_CODE>
              unknown:
                status:
                  ref: rule_statuses.<RULE_STATUS>
                reason_code: <REASON_CODE>
          require:
            all_of:
              - ref: requirements.<exported_requirement_name>
              - fact: <alias>.<fact_name>
                eq: <VALUE>
          result:
            satisfied:
              status:
                ref: rule_statuses.<RULE_STATUS>
              reason_code: <REASON_CODE>
            not_satisfied:
              status:
                ref: rule_statuses.<RULE_STATUS>
              reason_code: <REASON_CODE>
            unknown:
              status:
                ref: rule_statuses.<RULE_STATUS>
              reason_code: <REASON_CODE>

        # body form 2: ordered branches
        - id: <RULE_ID>
          select:
            from: <collection_name>
            as: <collection_alias>
            where:
              fact: <alias>.<fact_name>
              eq: <VALUE>
          branches:
            first_match:
              - when:
                  all_of:
                    - fact: <alias>.<fact_name>
                      eq: <VALUE>
                    - ref: requirements.<exported_requirement_name>
                result:
                  status:
                    ref: rule_statuses.<RULE_STATUS>
                  reason_code: <REASON_CODE>
                  condition: conditions.<CONDITION_NAME>   # only on conditional results
            unknown:
              result:
                status:
                  ref: rule_statuses.<RULE_STATUS>
                reason_code: <REASON_CODE>
            otherwise:
              result:
                status:
                  ref: rule_statuses.<RULE_STATUS>
                reason_code: <REASON_CODE>

# ============ SKELETON 3: shared definitions module (common/requirements.yaml and common/conditions.yaml) ============
dsl_version: "1.3"

module:
  id: <MODULE_ID>
  version: "<version-string>"

  exports:
    # in common/requirements.yaml: exported names are lowercase expressions
    <exported_requirement_name>:
      fact: <alias>.<fact_name>
      eq: <VALUE>
    <another_requirement_name>:
      any_of:
        - fact: <alias>.<fact_name>
          eq: <VALUE>
        - fact: <alias>.<fact_name>
          in:
            - <VALUE>
            - <VALUE>
    # in common/conditions.yaml: exported names are UPPERCASE, free-form parameter mappings
    # <CONDITION_NAME>:
    #   <parameter>: <value>
"""


class GeneratedFile(BaseModel):
    path: str
    content: str


class RulePackage(BaseModel):
    files: list[GeneratedFile]
    proposed_extensions: list[str] = []


SPEC_TEXT = (PROJECT_ROOT / "rules" / "README.md").read_text()
SCAFFOLD_FILES = {
    name: (RULES_DIR / name).read_text()
    for name in ("rule-statuses.yaml", "application-statuses.yaml")
}


def build_instructions() -> str:
    scaffold_blob = "\n\n".join(f"--- {name} (provided verbatim) ---\n{text}" for name, text in SCAFFOLD_FILES.items())
    if COMPILE_AND_EVALUATE:
        role = ("You are the rule authoring stage of an admissions rule extraction engine. From the "
                "extracted requirements and their vocabulary mapping, author a complete DSL 1.3 rule "
                "package that the deterministic compiler accepts.")
        contract = (
            "HARD REQUIREMENTS:\n"
            "- Produce exactly these files: " + ", ".join(sorted(EXPECTED_FILES)) + "\n"
            f"- The policy file is {POLICY_FILE} with policy id {POLICY_ID}, "
            f"applies_when application.study_level eq {STUDY_LEVEL}, and version \"{POLICY_VERSION}\".\n"
            "- The policy must define exactly the five rule ids from the vocabulary — no more, no fewer.\n"
            "- Every reason_code must be an exact key from the vocabulary's reason_codes.\n"
            "- Only use facts, values, and operators from the vocabulary; proposed_extensions must stay empty.\n"
            "- Every branch group needs unknown and otherwise results; requirements need satisfied, "
            "not_satisfied, and unknown results.\n"
            "- The policy resolution first_match must end with a when_no_recognized_rule case.\n"
            "- Record the handbook file and section titles in the policy sources field.\n"
            "- Unsupported requirements are out of scope: encode only supported ones."
        )
    else:
        role = ("You are the rule authoring stage of an admissions rule extraction engine. From the "
                "extracted requirements and their vocabulary mapping, author a PROPOSED DSL 1.3 rule "
                "package for the described program. The current engine vocabulary does not cover this "
                "program, so the package is for HUMAN REVIEW, not compilation.")
        contract = (
            "REQUIREMENTS FOR THE PROPOSED PACKAGE:\n"
            "- Produce exactly these files: " + ", ".join(sorted(EXPECTED_FILES)) + "\n"
            f"- The policy file is {POLICY_FILE} with policy id {POLICY_ID}, "
            f"applies_when application.study_level eq {STUDY_LEVEL}, and version \"0.1.0-proposed\".\n"
            "- You MAY propose new rule ids, facts, collections, enum values, and reason codes where "
            "this program needs them — follow the naming style of the existing vocabulary.\n"
            "- Every proposed vocabulary item that does not exist in the current engine must appear in "
            "proposed_extensions as one line each: '<kind>: <name> — <why needed>' (kinds: rule_id, "
            "collection, fact, enum_value, reason_code, condition, operator).\n"
            "- Keep every structural DSL rule: explicit satisfied/not_satisfied/unknown results, "
            "unknown and otherwise in every branch group, resolution first_match ending with "
            "when_no_recognized_rule, sources recorded with file/section/subsections.\n"
            "- Encode only what the handbook states; unclear or conflicting source text becomes "
            "MANUAL_REVIEW outcomes, not invented policy."
        )
    parts = [
        role,
        f"THE DSL SPECIFICATION:\n{SPEC_TEXT}",
        f"THE ENGINE VOCABULARY (JSON):\n{VOCAB_TEXT}",
        f"SCAFFOLD FILES ALREADY PRESENT IN THE PACKAGE — import them, never regenerate them:\n{scaffold_blob}",
        contract,
    ]
    parts.append(
        "FILE STRUCTURE SKELETON — structure only. Every <...> placeholder must be replaced "
        "using the vocabulary, the DSL spec, and the extracted requirements; the skeleton "
        f"carries no policy content:\n{POLICY_SKELETON}"
    )
    return "\n\n".join(parts)


def generate_package(error_feedback: str | None = None, previous: RulePackage | None = None) -> RulePackage:
    parts = [
        f"PROGRAM DESCRIPTION:\n{PROGRAM_DESCRIPTION}",
        f"EXTRACTED REQUIREMENTS (JSON):\n{json.dumps(requirements_json, indent=1)}",
        f"VOCABULARY MAPPING (JSON):\n{json.dumps(mapping_json, indent=1)}",
    ]
    if error_feedback is not None and previous is not None:
        previous_blob = "\n\n".join(f"--- {f.path} ---\n{f.content}" for f in previous.files)
        parts += [
            f"YOUR PREVIOUS ATTEMPT:\n{previous_blob}",
            f"THE COMPILER REJECTED IT WITH:\n{error_feedback}\n\nReturn the full corrected file set.",
        ]
    return ask(build_instructions(), "\n\n".join(parts), RulePackage)


def write_package(package: RulePackage) -> None:
    # The repair loop rewrites the package in place, so clear the rule files rather than
    # the whole run directory: artifacts/ from the earlier stages lives alongside them.
    for stale in RUN_DIR.glob("*.yaml"):
        stale.unlink()
    shutil.rmtree(RUN_DIR / "common", ignore_errors=True)
    (RUN_DIR / "common").mkdir(parents=True, exist_ok=True)
    for name, text in SCAFFOLD_FILES.items():
        (RUN_DIR / name).write_text(text)
    for file in package.files:
        if file.path not in EXPECTED_FILES:
            print(f"    skipping unexpected file: {file.path}")
            continue
        (RUN_DIR / file.path).write_text(file.content)


print("generation helpers ready; mode: "
      f"{'strict compile contract' if COMPILE_AND_EVALUATE else 'proposed package for review'}")


generation helpers ready; mode: strict compile contract


## Stage 5 — Compile with bounded repair

`RulesEngine.activate()` runs the real compiler: unknown facts, off-catalog reason codes, a wrong rule-id set, or a malformed resolution all reject the package with a typed error. On rejection, the error goes back to the model — at most 3 attempts per arm. The repair transcript is part of the result.

In [31]:
from app.rules_engine import RulesEngine

MAX_COMPILE_ATTEMPTS = 3


def compile_with_repair() -> dict:
    package = generate_package()
    for attempt in range(1, MAX_COMPILE_ATTEMPTS + 1):
        write_package(package)
        try:
            engine = RulesEngine.activate(RUN_DIR)
            print(f"attempt {attempt}: COMPILED -> {RUN_DIR.name}/")
            return {"engine": engine, "attempts": attempt, "error": None}
        except Exception as error:  # PolicyActivationError or spec validation errors
            code = getattr(error, "code", type(error).__name__)
            message = getattr(error, "safe_message", str(error))
            print(f"attempt {attempt}: REJECTED [{code}] {message}")
            if attempt == MAX_COMPILE_ATTEMPTS:
                return {"engine": None, "attempts": attempt, "error": f"{code}: {message}"}
            package = generate_package(error_feedback=f"{code}: {message}", previous=package)


def save_only() -> dict:
    package = generate_package()
    write_package(package)
    save_artifact("proposed-extensions.json", package.proposed_extensions)
    print(f"wrote {len(package.files)} files -> {RUN_DIR.name}/")
    print(f"{len(package.proposed_extensions)} proposed engine extensions")
    try:  # documented compile attempt, no repair — rejection expected and informative
        RulesEngine.activate(RUN_DIR)
        print("note: package unexpectedly compiles against the current engine")
    except Exception as error:
        code = getattr(error, "code", type(error).__name__)
        message = getattr(error, "safe_message", str(error))
        print(f"compile attempt (expected rejection): [{code}] {message}")
    return {"engine": None, "attempts": 1, "error": None}


outcome = (compile_with_repair if COMPILE_AND_EVALUATE else save_only)()


attempt 1: REJECTED [DUPLICATE_RULE_ID] The policy contains a duplicate rule identifier
attempt 2: COMPILED -> generate-rules-20260827-142343/


## Stage 6 — Gold evaluation: 16 saved applicants

Baseline = the **current hand-authored rules** evaluated on the same artifacts (the saved `decision-report.json` files were produced by policy v0.0.19 and are shown as a secondary reference). The evaluator pins the policy version, so each artifact's `program.policy.version` is patched in memory to match the engine being tested — nothing on disk changes. Agreement is measured on the final application status and on all five per-rule statuses.

In [32]:
if not COMPILE_AND_EVALUATE:
    print("skipped — no compile/eval in save-only mode; review the generated files and "
          "the proposed-extensions artifact instead")
else:
    from ruamel.yaml import YAML

    from app.io.artifact_io import load_facts_artifact
    from app.models.outcomes import EvaluationSucceeded


    def policy_version(rules_dir: Path) -> str:
        document = YAML(typ="safe").load((rules_dir / POLICY_FILE).read_text())
        return str(document["policy"]["version"])


    def with_policy_version(artifact, version: str):
        policy = artifact.program.policy.model_copy(update={"version": version})
        program = artifact.program.model_copy(update={"policy": policy})
        return artifact.model_copy(update={"program": program})


    gold_runs = sorted(d for d in RUNS_DIR.iterdir() if (d / "application-facts.json").exists())
    artifacts = {d.name: load_facts_artifact(d / "application-facts.json") for d in gold_runs}
    saved_reports = {
        d.name: json.loads((d / "decision-report.json").read_text()).get("application_status", "—")
        for d in gold_runs
    }


    def decide_all(engine: RulesEngine, version: str) -> dict[str, dict]:
        decisions = {}
        for name, artifact in artifacts.items():
            result_or_failure = engine.evaluate(with_policy_version(artifact, version))
            if isinstance(result_or_failure, EvaluationSucceeded):
                result = result_or_failure.result
                decisions[name] = {
                    "status": result.application_status.value,
                    "rules": {rule.rule_id.value: rule.status.value for rule in result.rules},
                }
            else:
                decisions[name] = {"status": f"FAILED:{result_or_failure.failure.code}", "rules": {}}
        return decisions


    baseline = decide_all(RulesEngine.activate(RULES_DIR), policy_version(RULES_DIR))
    generated = (decide_all(outcome["engine"], policy_version(RUN_DIR))
                 if outcome["engine"] else None)

    width = max(len(name) for name in artifacts)
    header = f"{'applicant':<{width}}  {'saved report':<20} {'hand-authored':<22}"
    if generated:
        header += f" {'generated':<22}"
    print(header)
    print("-" * len(header))
    for name in artifacts:
        row = f"{name:<{width}}  {saved_reports[name]:<20} {baseline[name]['status']:<22}"
        if generated:
            status = generated[name]["status"]
            marker = "=" if status == baseline[name]["status"] else "≠"
            row += f" {marker + ' ' + status:<22}"
        print(row)


ValueError: max() iterable argument is empty

In [ ]:
if not COMPILE_AND_EVALUATE:
    print("skipped — no compile/eval in save-only mode; review the generated files and "
          "the proposed-extensions artifact instead")
else:
    summary = {"model": MODEL, "program_description": PROGRAM_DESCRIPTION,
               "sections_retrieved": len(opened),
               "requirements_extracted": len(requirement_set.requirements),
               "requirements_supported": len(supported),
               "requirements_unsupported": len(unsupported),
               "compiled": bool(outcome["engine"]),
               "attempts": outcome["attempts"],
               "error": outcome["error"]}

    total = len(artifacts)
    if generated:
        status_hits = sum(generated[n]["status"] == baseline[n]["status"] for n in artifacts)
        rule_hits = sum(generated[n]["rules"] == baseline[n]["rules"] for n in artifacts)
        print(f"compiled:           yes, after {outcome['attempts']} attempt(s)")
        print(f"status agreement:   {status_hits}/{total}")
        print(f"per-rule agreement: {rule_hits}/{total}")
        summary["status_agreement"] = f"{status_hits}/{total}"
        summary["per_rule_agreement"] = f"{rule_hits}/{total}"
        summary["decisions"] = generated
    else:
        print(f"compiled:           no, after {outcome['attempts']} attempt(s)")
        print(f"error:              {outcome['error']}")

    summary["baseline"] = baseline
    save_artifact("summary.json", summary)
    print(f"\neverything from this run -> {RUN_DIR}")
